# Colab GPU Experiment Runner

This notebook is built for shared Google Drive runs. Open it in one or more Colab GPU sessions, point them at the same repo checkout, and choose one of three modes:

- `prepare`: build shared RecBole caches and manifests once
- `worker`: claim pending `(category, model)` jobs from the shared `run/` directory
- `finalize`: generate the paper plots after all jobs complete

The worker mode is the scalable one: multiple Colab sessions can run it in parallel without duplicating jobs.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = os.environ.get('REPO_URL', 'https://github.com/andersvestrum/carbon-aware-recsys.git')
REPO_BRANCH = os.environ.get('REPO_BRANCH', '6-neumfs-sharp-curve')
AUTO_CLONE_IF_MISSING = True
DEFAULT_CLONE_ROOT = Path('/content/drive/MyDrive')

# If auto-discovery misses your Drive layout, set this manually.
MANUAL_PROJECT_ROOT = None  # e.g. '/content/drive/MyDrive/path/to/carbon-aware-recsys'

def is_project_root(path):
    return (path / 'scripts' / '05_run_full_experiment.py').exists()

def clone_or_update_repo(target):
    target = target.expanduser().resolve()
    target.parent.mkdir(parents=True, exist_ok=True)
    if (target / '.git').exists():
        subprocess.run(['git', '-C', str(target), 'fetch', 'origin', REPO_BRANCH], check=True)
        subprocess.run(['git', '-C', str(target), 'checkout', REPO_BRANCH], check=True)
        subprocess.run(['git', '-C', str(target), 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)
        return target

    if target.exists() and any(target.iterdir()):
        raise FileExistsError(f'{target} exists but is not a git checkout. Set MANUAL_PROJECT_ROOT explicitly.')

    if target.exists():
        target.rmdir()

    subprocess.run([
        'git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(target)
    ], check=True)
    return target

def discover_project_root():
    env_root = os.environ.get('PROJECT_ROOT')
    if env_root:
        candidate = Path(env_root).expanduser().resolve()
        if is_project_root(candidate):
            return candidate

    if MANUAL_PROJECT_ROOT is not None:
        candidate = Path(MANUAL_PROJECT_ROOT).expanduser().resolve()
        if is_project_root(candidate):
            return candidate

    cwd = Path.cwd().resolve()
    direct_candidates = [
        cwd,
        *cwd.parents,
        Path('/content/carbon-aware-recsys'),
        Path('/content/drive/MyDrive/carbon-aware-recsys'),
    ]
    for candidate in direct_candidates:
        if is_project_root(candidate):
            return candidate.resolve()

    for base in [Path('/content'), Path('/content/drive/MyDrive')]:
        if not base.exists():
            continue
        for candidate in base.glob('**/carbon-aware-recsys'):
            if is_project_root(candidate):
                return candidate.resolve()

    return None

PROJECT_ROOT = discover_project_root()
if PROJECT_ROOT is None and AUTO_CLONE_IF_MISSING:
    clone_root = DEFAULT_CLONE_ROOT if DEFAULT_CLONE_ROOT.exists() else Path('/content')
    clone_target = clone_root / 'carbon-aware-recsys'
    PROJECT_ROOT = clone_or_update_repo(clone_target)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find or clone the repo. Set MANUAL_PROJECT_ROOT to the repo root.")

RUN_DIR = PROJECT_ROOT / 'run'
PROJECT_ROOT, RUN_DIR

In [ ]:
%cd {PROJECT_ROOT}
%pip install -q -r requirements.txt

In [ ]:
import torch

GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
GPU_MEMORY_GB = (
    torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    if torch.cuda.is_available()
    else 0.0
)

if GPU_MEMORY_GB >= 35:
    AUTO_TRAIN_BATCH_SIZE = 8192
    AUTO_EVAL_BATCH_SIZE = 16384
    AUTO_USER_BATCH_SIZE = 1024
elif GPU_MEMORY_GB >= 20:
    AUTO_TRAIN_BATCH_SIZE = 6144
    AUTO_EVAL_BATCH_SIZE = 12288
    AUTO_USER_BATCH_SIZE = 768
elif GPU_MEMORY_GB >= 14:
    AUTO_TRAIN_BATCH_SIZE = 4096
    AUTO_EVAL_BATCH_SIZE = 8192
    AUTO_USER_BATCH_SIZE = 512
else:
    AUTO_TRAIN_BATCH_SIZE = 2048
    AUTO_EVAL_BATCH_SIZE = 4096
    AUTO_USER_BATCH_SIZE = 256

print({'gpu': GPU_NAME, 'gpu_memory_gb': round(GPU_MEMORY_GB, 2)})
print({
    'train_batch_size': AUTO_TRAIN_BATCH_SIZE,
    'eval_batch_size': AUTO_EVAL_BATCH_SIZE,
    'user_batch_size': AUTO_USER_BATCH_SIZE,
})

In [ ]:
MODE = 'worker'  # choose: 'prepare', 'worker', 'finalize'
WORKER_NAME = 'colab-gpu-1'  # give each Colab worker a unique name

CATEGORIES = ['electronics', 'home_and_kitchen', 'sports_and_outdoors']
MODELS = ['BPR', 'NeuMF', 'LightGCN']

TOP_K_CANDIDATES = 100
TOP_K_RERANK = 10
USER_BATCH_SIZE = AUTO_USER_BATCH_SIZE
TRAIN_BATCH_SIZE = AUTO_TRAIN_BATCH_SIZE
EVAL_BATCH_SIZE = AUTO_EVAL_BATCH_SIZE
LEARNING_RATE = 1e-3
EPOCHS = 50
EVAL_STEP = 10

MAX_USERS = None

PREPARE_CARBON = False
FORCE = False
FORCE_CARBON = False
SKIP_LLM = False
LLM_CACHE_ONLY = False
FINALIZE_WHEN_DONE = False  # set True only on one worker if you want auto-finalization


In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    'scripts/05_run_full_experiment.py',
    '--run-dir', str(RUN_DIR),
    '--interim-dir', str(PROJECT_ROOT / 'data' / 'interim'),
    '--top-k-candidates', str(TOP_K_CANDIDATES),
    '--top-k-rerank', str(TOP_K_RERANK),
    '--user-batch-size', str(USER_BATCH_SIZE),
    '--train-batch-size', str(TRAIN_BATCH_SIZE),
    '--eval-batch-size', str(EVAL_BATCH_SIZE),
    '--learning-rate', str(LEARNING_RATE),
    '--epochs', str(EPOCHS),
    '--eval-step', str(EVAL_STEP),
    '--categories', *CATEGORIES,
    '--models', *MODELS,
]

if MAX_USERS is not None:
    cmd.extend(['--max-users', str(MAX_USERS)])
if PREPARE_CARBON:
    cmd.append('--prepare-carbon')
if FORCE:
    cmd.append('--force')
if FORCE_CARBON:
    cmd.append('--force-carbon')
if SKIP_LLM:
    cmd.append('--skip-llm')
if LLM_CACHE_ONLY:
    cmd.append('--llm-cache-only')

if MODE == 'prepare':
    cmd.append('--prepare-only')
elif MODE == 'worker':
    cmd.extend([
        '--claim-jobs',
        '--skip-cache-prepare',
        '--skip-plotting',
        '--worker-name', WORKER_NAME,
    ])
    if FINALIZE_WHEN_DONE:
        cmd.append('--finalize-when-done')
elif MODE == 'finalize':
    cmd.append('--plots-only')
else:
    raise ValueError(f'Unsupported MODE: {MODE}')

print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
import json
import pandas as pd
from IPython.display import Image, display

summary_path = RUN_DIR / 'results' / 'dataset_summary.csv'
best_points_path = RUN_DIR / 'results' / 'paper_best_pareto_points.csv'
manifest_path = RUN_DIR / 'results' / 'paper_plot_manifest.json'
state_dir = RUN_DIR / 'results' / '_job_state'

if summary_path.exists():
    display(pd.read_csv(summary_path))
if state_dir.exists():
    status_rows = []
    for path in sorted(state_dir.glob('*.done.json')):
        payload = json.loads(path.read_text())
        status_rows.append({'state': 'done', **payload})
    for path in sorted(state_dir.glob('*.failed.json')):
        payload = json.loads(path.read_text())
        status_rows.append({'state': 'failed', **payload})
    for path in sorted(state_dir.glob('*.running.json')):
        payload = json.loads(path.read_text())
        status_rows.append({'state': 'running', **payload})
    if status_rows:
        display(pd.DataFrame(status_rows).sort_values(['state', 'category', 'model'], na_position='last'))
if best_points_path.exists():
    display(pd.read_csv(best_points_path))
if manifest_path.exists():
    with manifest_path.open() as handle:
        manifest = json.load(handle)
    print(f"Generated {len(manifest['figures'])} figures")
    for figure_path in manifest['figures'][:6]:
        display(Image(filename=figure_path))